In [12]:
import pandas as pd
import numpy as np
import random


# CONFIGURAÇÕES DA SIMULAÇÃO

In [13]:
NUM_USUARIOS = 500
NUM_AVALIACOES = 7000
ARQUIVO_ARTIGOS = 'dados_tratados.csv'
ARQUIVO_SAIDA = 'avaliacoes_simuladas.csv'

# PASSO 1: Carregar os dados dos artigos

In [14]:
try:
    df_artigos = pd.read_csv(ARQUIVO_ARTIGOS)
except FileNotFoundError:
    print(f"Erro: Arquivo '{ARQUIVO_ARTIGOS}' não encontrado. Certifique-se de que ele está no diretório correto.")
    # Se o arquivo não for encontrado, o script para aqui.
    exit()

# Identifica quais são as colunas de categoria (todas após 'abstracts')

In [15]:
colunas_de_categoria = df_artigos.columns[3:].tolist()


#  PASSO 2: Definir os "Arquétipos" de Interesse
# Agrupamos categorias que fazem sentido juntas.

In [16]:
arquétipos = {
    "Especialista em IA e Machine Learning": [
        'cs.AI', # Inteligência Artificial
        'cs.LG', # Aprendizado de Máquina
        'cs.NE', # Computação Neural e Evolutiva
        'cs.CL'  # Linguística Computacional
    ],
    "Engenheiro de Sistemas e Hardware": [
        'cs.AR', # Arquitetura de Computadores
        'cs.OS', # Sistemas Operacionais
        'cs.DC', # Computação Distribuída, Paralela e em Cluster
        'cs.SY'  # Sistemas e Controle
    ],
    "Desenvolvedor de Software e Segurança": [
        'cs.SE', # Engenharia de Software
        'cs.PL', # Linguagens de Programação
        'cs.CR', # Criptografia e Segurança
        'cs.DB'  # Banco de Dados
    ],
    "Pesquisador em Teoria da Computação": [
        'cs.CC', # Complexidade Computacional
        'cs.GT', # Ciência da Computação e Teoria dos Jogos
        'cs.LO', # Lógica na Ciência da Computação
        'cs.DS'  # Estruturas de Dados e Algoritmos
    ],
    "Especialista em Robótica e Visão": [
        'cs.RO', # Robótica
        'cs.CV', # Visão Computacional
        'cs.AI', # Inteligência Artificial
        'cs.SY'  # Sistemas e Controle
    ],
    "Cientista de Dados e Informação": [
        'cs.IR', # Recuperação da Informação
        'cs.DB', # Banco de Dados
        'cs.DM', # Mineração de Dados
        'cs.SI'  # Análise de Redes Sociais e Informação
    ],
    "Especialista em Interação e Gráficos": [
        'cs.HC', # Interação Humano-Computador
        'cs.CG', # Computação Gráfica
        'cs.MM', # Multimídia
        'cs.GR'  # Gráficos
    ]
}

# Filtra os arquétipos para usar apenas categorias que realmente existem no seu CSV

In [17]:
arquétipos_validos = {}
for nome, categorias in arquétipos.items():
    categorias_presentes = [cat for cat in categorias if cat in colunas_de_categoria]
    if categorias_presentes:
        arquétipos_validos[nome] = categorias_presentes

if not arquétipos_validos:
    print("Nenhuma das categorias nos arquétipos foi encontrada no seu CSV. Usando todas as categorias.")
    arquétipos_validos = {"Geral": colunas_de_categoria}

# PASSO 3: Criar os Perfis dos Usuários

In [18]:
perfis_usuarios = {}
lista_arquétipos = list(arquétipos_validos.keys())

for user_id in range(1, NUM_USUARIOS + 1):
    # Sorteia um arquétipo para o usuário
    tipo_perfil = random.choice(lista_arquétipos)
    # Associa as categorias daquele arquétipo ao usuário
    perfis_usuarios[user_id] = arquétipos_validos[tipo_perfil]

print(f"{len(perfis_usuarios)} perfis de usuários criados com sucesso.")

500 perfis de usuários criados com sucesso.


# PASSO 4: Gerar as Avaliações com Base nos Perfis

In [19]:
lista_avaliacoes = []

for _ in range(NUM_AVALIACOES):
    # Escolhe um usuário aleatório para fazer a avaliação
    usuario_id = random.choice(list(perfis_usuarios.keys()))
    categorias_preferidas = perfis_usuarios[usuario_id]
    
    # 85% de chance de avaliar um artigo de sua área de interesse
    if random.random() < 0.85:
        # AVALIAÇÃO POSITIVA (dentro do perfil)
        
        # Filtra os artigos que pertencem a PELO MENOS UMA das categorias preferidas do usuário
        mascara_preferidos = df_artigos[categorias_preferidas].any(axis=1)
        artigos_candidatos = df_artigos[mascara_preferidos]
        
        if not artigos_candidatos.empty:
            artigo_escolhido = artigos_candidatos.sample(1)
            # Atribui uma nota alta (4 ou 5)
            nota = random.choice([4, 5])
        else:
            # Se não houver artigos no perfil, pega um aleatório (caso raro)
            artigo_escolhido = df_artigos.sample(1)
            nota = 3

    else:
        # AVALIAÇÃO DE EXPLORAÇÃO (fora do perfil)
        
        # Filtra artigos que NÃO pertencem a NENHUMA das categorias preferidas
        mascara_preferidos = df_artigos[categorias_preferidas].any(axis=1)
        artigos_candidatos = df_artigos[~mascara_preferidos]

        if not artigos_candidatos.empty:
            artigo_escolhido = artigos_candidatos.sample(1)
            # Atribui uma nota baixa ou neutra (1, 2 ou 3)
            nota = random.choice([1, 2, 3])
        else:
            # Se todos os artigos forem do perfil do usuário, pega um aleatório
            artigo_escolhido = df_artigos.sample(1)
            nota = 3
    
    # Adiciona a avaliação gerada à lista
    artigo_id = artigo_escolhido['ids'].iloc[0]
    lista_avaliacoes.append({
        'user_id': usuario_id,
        'artigo_id': artigo_id,
        'rating': nota
    })

# PASSO 5: Criar e Salvar o DataFrame Final


In [20]:
df_avaliacoes_finais = pd.DataFrame(lista_avaliacoes)


# Remove duplicatas, caso o mesmo usuário avalie o mesmo artigo duas vezes

In [21]:
df_avaliacoes_finais = df_avaliacoes_finais.drop_duplicates(subset=['user_id', 'artigo_id'], keep='last')


In [22]:
df_avaliacoes_finais.to_csv(ARQUIVO_SAIDA, index=False)

print(f"\nArquivo '{ARQUIVO_SAIDA}' gerado com sucesso!")
print(f"Total de avaliações únicas geradas: {len(df_avaliacoes_finais)}")
print("\n--- Amostra das Avaliações Geradas ---")
print(df_avaliacoes_finais.head())


Arquivo 'avaliacoes_simuladas.csv' gerado com sucesso!
Total de avaliações únicas geradas: 6993

--- Amostra das Avaliações Geradas ---
   user_id   artigo_id  rating
0      420  2303.08409       4
1       59  1908.10870       3
2      310  1612.01543       5
3       64  2310.03616       4
4      244  2006.08666       4
